![DBAcademy](./Includes/images/common/db-academy.png)

# Demo - Build the Silver Layer

## Overview

In the previous demo, you added four bronze sources to your Lakeflow Designer visual data prep. In this demo, you will build the **silver layer** by cleaning and transforming each bronze table into a silver table, using a different authoring style for each:

- The **Prepare and Select** operators (point-and-click).
- **Genie Code** (AI-generated transformations).
- The **SQL** operator (custom code).

Each silver table will be materialized in Unity Catalog so the gold layer (next demo) can read from it.

## Learning Objectives

By the end of this demo, you will be able to:

- Use the **Prepare** and **Select** operators to cast, rename, and drop columns.
- Use **Genie Code** to describe transformations in natural language and let the AI generate the operators.
- Use the **SQL** operator to parse semi-structured JSON data with `parse_json`, `VARIANT`, and `variant_explode_outer`.
- Materialize each silver table in Unity Catalog using the **Output** operator.

## REQUIRED PREREQUISITES
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">PREREQUISITES</strong>
  <div style="color:#333;">
You must complete <strong>02 - Lakeflow Designer Overview and Source Layer</strong> before starting this demo.

Before continuing, confirm that the following four bronze tables are on your Lakeflow Designer canvas:

- <strong>sessions_bronze</strong>
- <strong>participants_bronze</strong>
- <strong>feedback_bronze</strong>
- <strong>participant_lookup_bronze</strong>
  </div>
</div>

## About the Silver Layer

The silver layer is where bronze data gets cleaned up: type casting, renaming for clarity, and dropping columns we do not need downstream.

We will build one silver table per bronze table.

<!--
  =============================================================================
  HOW TO GRAY OUT NODES
  =============================================================================
  Add class="dim" (alongside any existing classes) to any element you want to
  fade into the background.
  =============================================================================
-->
<div class="pipe-wrap">

<style>
.pipe-wrap {
    max-width: 920px;
    margin: 0 auto;
    font-family: sans-serif;
    color: #0b2026;
    padding: 8px;
    box-sizing: border-box;
}

/* Add class="dim" to any element to fade it into the background */
.dim {
    opacity: 0.30;
    filter: grayscale(100%);
}

.pipe-header {
    background: #1B5162;
    color: white;
    border-radius: 10px;
    padding: 22px 26px;
    text-align: center;
    margin-bottom: 22px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.10);
}

.pipe-title {
    font-size: 20pt;
    font-weight: 700;
    color: white;
    margin: 0 0 10px 0;
}

.pipe-subtitle {
    font-size: 13pt;
    color: white;
    opacity: 0.94;
    line-height: 1.5;
    margin: 0;
    display: inline-block;
    text-align: left;
    padding-left: 24px;
}

.pipe-subtitle li { margin-bottom: 4px; }
.pipe-subtitle li:last-child { margin-bottom: 0; }
.pipe-subtitle strong { color: #FFAB00; font-weight: 700; }

.pipe-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    column-gap: 14px;
    row-gap: 8px;
    align-items: stretch;
    justify-items: stretch;
}

.pipe-arrow-cell {
    display: flex;
    justify-content: center;
    align-items: center;
}

.pipe-node {
    width: 100%;
    height: 100%;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    padding: 14px 10px 12px 10px;
    text-align: center;
    position: relative;
    box-sizing: border-box;
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
}

.pipe-node::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 6px;
    border-top-left-radius: 8px;
    border-top-right-radius: 8px;
}

.pipe-node.source::before      { background: #00A972; }
.pipe-node.bronze::before      { background: #CD7F32; }
.pipe-node.silver::before      { background: #90A5B1; }
.pipe-node.gold::before        { background: #FFAB00; }

/* Bonus card: dashed border instead of solid shadow */
.pipe-node.bonus {
    border: 2px dashed #B57F00;
    background: rgba(255, 171, 0, 0.06);
    box-shadow: none;
}

.bonus-badge {
    position: absolute;
    top: -10px;
    right: 8px;
    background: #B57F00;
    color: white;
    font-size: 8pt;
    font-weight: 700;
    letter-spacing: 1px;
    padding: 3px 8px;
    border-radius: 4px;
    text-transform: uppercase;
    z-index: 2;
}

.pipe-icon {
    width: 30px;
    height: 30px;
    margin: 4px auto 6px auto;
    display: block;
}

.pipe-node.source .pipe-icon      { color: #00A972; }
.pipe-node.bronze .pipe-icon      { color: #CD7F32; }
.pipe-node.silver .pipe-icon      { color: #5A6F77; }
.pipe-node.gold .pipe-icon        { color: #B57F00; }

.pipe-node-label {
    font-size: 9pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    margin: 0 0 4px 0;
}

.pipe-node.source .pipe-node-label      { color: #00A972; }
.pipe-node.bronze .pipe-node-label      { color: #CD7F32; }
.pipe-node.silver .pipe-node-label      { color: #5A6F77; }
.pipe-node.gold .pipe-node-label        { color: #B57F00; }

.pipe-node-name {
    font-size: 11pt;
    font-weight: 700;
    color: #0b2026;
    word-break: break-word;
    line-height: 1.3;
    margin: 0 0 6px 0;
}

.pipe-node-desc {
    font-size: 12pt;
    font-weight: 400;
    color: #5E7077;
    line-height: 1.4;
    margin: 0;
}

.pipe-arrow {
    width: 22px;
    height: 28px;
    color: #5E7077;
}

.volume-wrapper {
    border: 2px dashed #1B5162;
    border-radius: 10px;
    padding: 10px 8px 10px 8px;
    background: rgba(27,81,98,0.04);
    box-sizing: border-box;
    height: 100%;
    display: flex;
    flex-direction: column;
}

.volume-label {
    font-size: 8.5pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: #1B5162;
    text-align: center;
    margin: 0 0 8px 0;
}

.upstream-wrapper {
    border: 2px dashed #5E7077;
    border-radius: 10px;
    padding: 10px 14px;
    background: rgba(94,112,119,0.06);
    box-sizing: border-box;
    height: 100%;
    display: flex;
    flex-direction: column;
}

.upstream-label {
    font-size: 10pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: #5E7077;
    text-align: center;
    margin: 0 0 8px 0;
}

.upstream-body {
    display: flex;
    align-items: center;
    gap: 14px;
    flex: 1;
    padding: 4px 8px;
}

.upstream-icon {
    width: 36px;
    height: 36px;
    color: #5E7077;
    flex-shrink: 0;
}

.upstream-title {
    font-size: 14pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 4px;
    line-height: 1.3;
}

.upstream-desc {
    font-size: 12pt;
    color: #5E7077;
    line-height: 1.4;
}

.silver-to-gold {
    display: flex;
    flex-direction: column;
    align-items: center;
    margin: 14px 0 18px 0;
}

.silver-to-gold svg {
    width: 30px;
    height: 38px;
    color: #5E7077;
}

.silver-to-gold-caption {
    font-size: 13pt;
    font-weight: 700;
    color: #1B5162;
    margin-top: 4px;
}

.gold-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    column-gap: 14px;
    margin-top: 0;
}
</style>

<div class="pipe-header">
  <h3 class="pipe-title">The Visual Workflow You Will Build</h3>
  <ul class="pipe-subtitle">
    <li>Import a local CSV file into a volume.</li>
    <li>Bronze gets cleaned into <strong>silver</strong>.</li>
    <li>Two gold tables are derived from silver. A bonus AI-powered table is offered as homework.</li>
  </ul>
</div>

<div class="pipe-grid">

  <!-- Row 1: Source (CSV) + Upstream ETL banner -->
  <div style="grid-column: 1; grid-row: 1;">
    <div class="volume-wrapper">
      <div class="volume-label">Manual File Upload to a Volume</div>
      <div class="pipe-node source">
        <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><path d="M5 2h11l4 4v15a1 1 0 0 1-1 1H5a1 1 0 0 1-1-1V3a1 1 0 0 1 1-1z"/><path d="M16 2v4h4"/><text x="12" y="19" text-anchor="middle" font-family="sans-serif" font-size="10" font-weight="700" stroke="none" fill="currentColor">CSV</text></svg>
        <div class="pipe-node-label">CSV File</div>
        <p class="pipe-node-name">how_participant_lookup_week_1.csv</p>
        <p class="pipe-node-desc">Uploaded participant directory</p>
      </div>
    </div>
  </div>

  <div style="grid-column: 2 / span 3; grid-row: 1;">
    <div class="upstream-wrapper">
      <div class="upstream-label">Upstream workflow &middot; Runs on a Schedule</div>
      <div class="upstream-body">
        <svg class="upstream-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">
          <path d="M21 12a9 9 0 1 1-3.51-7.12"/>
          <polyline points="21 4 21 9 16 9"/>
        </svg>
        <div>
          <div class="upstream-title">Source workflow managed by another team</div>
          <div class="upstream-desc">Another team ingests the meeting source data as bronze tables on a schedule, refreshing the tables with new sessions, attendees, and feedback. You will keep silver and gold updated.</div>
        </div>
      </div>
    </div>
  </div>

  <!-- Row 2: arrows from CSV and Upstream ETL down to bronze -->
  <div class="pipe-arrow-cell" style="grid-column: 1; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 2; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 3; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 4; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>

  <!-- Row 3: Bronze tables -->
  <div style="grid-column: 1; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">participant_lookup_bronze</p>
      <p class="pipe-node-desc">Lookup ingested from the CSV</p>
    </div>
  </div>
  <div style="grid-column: 2; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">sessions_bronze</p>
      <p class="pipe-node-desc">Raw session metadata, one row per session</p>
    </div>
  </div>
  <div style="grid-column: 3; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">participants_bronze</p>
      <p class="pipe-node-desc">Attendance log, one row per attendee per session</p>
    </div>
  </div>
  <div style="grid-column: 4; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">feedback_bronze</p>
      <p class="pipe-node-desc">Survey responses stored as a JSON array per session</p>
    </div>
  </div>

  <!-- Row 4: Bronze -> Silver arrows -->
  <div class="pipe-arrow-cell" style="grid-column: 1; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 2; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 3; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 4; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>

  <!-- Row 5: Silver tables -->
  <div style="grid-column: 1; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">participant_lookup_silver</p>
      <p class="pipe-node-desc">Email and region split into parts</p>
    </div>
  </div>
  <div style="grid-column: 2; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">sessions_silver</p>
      <p class="pipe-node-desc">Date typed, how_name renamed to session_name</p>
    </div>
  </div>
  <div style="grid-column: 3; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">participants_silver</p>
      <p class="pipe-node-desc">Timestamps typed, duration_in_minutes added</p>
    </div>
  </div>
  <div style="grid-column: 4; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">feedback_silver</p>
      <p class="pipe-node-desc">JSON exploded, one row per response</p>
    </div>
  </div>

</div>

<!-- Single centered arrow between silver and gold -->
<div class="silver-to-gold">
  <svg width="30" height="38" viewBox="0 0 30 38" fill="none" aria-hidden="true"><line x1="15" y1="2" x2="15" y2="26" stroke="currentColor" stroke-width="3" stroke-linecap="round"/><polygon points="4,26 26,26 15,38" fill="currentColor"/></svg>
  <div class="silver-to-gold-caption">Join and aggregate</div>
</div>

<!-- Gold row: 2 main gold tables + 1 bonus -->
<div class="gold-grid">
  <div class="pipe-node gold dim">
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_session_summary</p>
    <p class="pipe-node-desc">Session metadata with attendance metrics</p>
  </div>
  <div class="pipe-node gold dim">
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_feedback_summary</p>
    <p class="pipe-node-desc">Session-level average ratings</p>
  </div>
  <div class="pipe-node gold bonus dim">
    <div class="bonus-badge">Bonus / Homework</div>
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_feedback_sentiment</p>
    <p class="pipe-node-desc">Free-form responses tagged with ai_sentiment</p>
  </div>
</div>

</div>


## A. Create the `sessions_silver` Table

For **sessions_silver**, we need to:

- Convert the **event_date** column from a `STRING` to a `DATE`.
- Rename **how_name** to **session_name**.
- Drop the **_rescued_data** column.
- Keep all other columns as is.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">

We will use point-and-click operators for this first silver table. Later we will use **Genie Code** (AI) and the **SQL** operator to do the same kind of work in different ways.

  </div>
</div>


### A1. Sessions Silver Prepare and Select

1. Select the **sessions_bronze** node and view the data in the **Preview** pane.


#### A1.1 Prepare Operator

1. Navigate to the circle on the right of the node, then select **+** > **Prepare** to add the operator.

2. Double-click the **Prepare** operator node to open the configuration panel.

3. Select **Add action** > **Change type**

    a. **Column** : `event_date`

    b. **New type** : `date`

    c. **On parse error** : `Set to null`

    d. Rename the node: `sessions_preparation`

    e. Click **Apply**

    > **Note:** Clicking **Apply** saves the configuration on the node and runs the individual operator. It does not run the entire workflow or create any tables yet. The `sessions_silver` table will be created when you run the full workflow in the **Output** node.

3. Review the **Input** and **Output** preview pane and confirm:
    - **event_date** is now a `DATE` type.

#### A1.2 Select Operator
1. On the **sessions_prepration** node select **+** > **Select** to add the operator.
    
    a. **Rename how_name to session_name:**
    - Locate the **how_name** column in the operator panel.
    -  Rename it to `session_name` using the rename action.

    b. **Drop the _rescued_data column:**
    - Locate the **_rescued_data** column.
    - Remove it from the output (uncheck it or use the drop/remove action).

    c. You can also reorder the columns if you'd like. Set the following as columns 1 & 2:
    - Column 1: **meeting_uuid**
    - Column 2: **how_name | session_name**

    d. Rename the operator to `sessions_columns`.

    e. Click **Apply**.
 
6. View the **Output** table in the preview pane and confirm:
    - **event_date** is now a `DATE` type.
    - **how_name** is now called **session_name**.
    - **_rescued_data** is no longer in the column list.
    - The row count matches **sessions_bronze** (41).

7. In the **Output** pane at the bottom right, notice that only **1000** rows were scanned. You can modify that if you'd like.



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

Lakeflow Designer previews run on a sample of up to 1,000 rows by default to keep the UI responsive while you build and validate steps.

Change the preview setting when the sample is misleading, especially for joins, filters, or value discovery where important matches or values might not appear in the sampled rows.

If you need to validate against all available data, switch the output pane from **Rows scanned: Limit** to **Rows scanned: Max**.

**The output pane** documentation: [AWS](https://docs.databricks.com/aws/en/designer/what-is-lakeflow-designer#the-output-pane) | 
[Azure](https://learn.microsoft.com/en-us/azure/databricks/designer/what-is-lakeflow-designer#output) |
[GCP](https://docs.databricks.com/gcp/en/designer/what-is-lakeflow-designer#the-output-pane)

  </div>
</div>


#### Checkpoint - Sessions Silver Prepare and Select

![sessions_silver_transform](./Includes/images/visual_etl_pipeline/checkpoint-session-siver-prepare-select.png)

### A2. Create the `sessions_silver` Table

1. Select your **sessions_columns** node.

2. Select **+** > **Output** to add the output operator. Then create the table.

3. Configure the **Output** panel with the following:
    | Setting | Value |
    |---|---|
    | **Table name** | `sessions_silver` |
    | **Catalog** | `learn_no_code_etl_designer_USERNAME` |
    | **Schema** | `designer_meeting` |
    > **NOTE:** If the **designer_meeting** schema does not appear in the dropdown, select the **x** icon to reload the Unity Catalog schemas.

4. Rename the Output node to `sessions_silver_table`.

5. Select **Run** to create the table and confirm it was successful.

6. Verify the table was created in Unity Catalog:
    - Select the **Catalog** icon ![Catalog Icon](./Includes/images/common/catalog-icon.png)
    - Right-click on the **learn_no_code_etl_designer_USERNAME** catalog > **Open in Catalog Explorer**

    - Navigate to the **designer_meeting** schema (refresh if necessary).

    - Confirm you see the **sessions_silver** table in the table list.

    - Select **sessions_silver** to inspect the schema then the **Sample Data**.

    - Confirm:
        - **event_date** is a `DATE` type.
        - **session_name** column exists (renamed from **how_name**).
        - **_rescued_data** column is gone.

#### Checkpoint - Sessions Silver Table

![sessions_silver_table](./Includes/images/visual_etl_pipeline/checkpoint-sessions-silver-tbl.png)

## B. Create the `participants_silver` Table

This time we will use **Genie Code** (AI) to build the silver table in a single prompt. Instead of dragging and configuring operators by hand, we describe what we want and the AI generates the nodes for us.

In the data we need to:

- Drop the **_rescued_data** column.
- Convert **join_time** and **leave_time** from `STRING` to `TIMESTAMP`.
- Drop all rows with a `NULL` id.
- Add a new column **duration_in_minutes** that calculates how long each participant stayed in the session, in whole minutes.
    - If **leave_time** is before **join_time** (negative duration), set **duration_in_minutes** to `NULL`.
- Keep all other columns as is.

1. Select the **participants_bronze** node and view the data in the **Preview** pane.

2. In the **Genie Code** prompt bar, enter the following prompt:


<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; margin: 8px 0; font-family: sans-serif; max-width: 1100px; overflow: hidden;">
  <div style="display: flex; justify-content: space-between; align-items: center; background: #EEEDE9; padding: 6px 12px; border-bottom: 1px solid #d9d7d2;">
    <span style="font-size: 10pt; font-weight: 700; color: #0b2026; letter-spacing: 1px; text-transform: uppercase;">Genie Code Prompt</span>
    <button
      type="button"
      aria-label="Copy Genie Code prompt to clipboard"
      onmouseover="this.style.background='#143F4D';"
      onmouseout="this.style.background='#1B5162';"
      onclick="
        var text = this.parentElement.nextElementSibling.innerText;
        var btn = this;
        var ok = false;
        try {
          var ta = document.createElement('textarea');
          ta.value = text;
          ta.style.position = 'fixed';
          ta.style.top = '0';
          ta.style.left = '0';
          ta.style.opacity = '0';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          ok = document.execCommand('copy');
          document.body.removeChild(ta);
        } catch (e) { ok = false; }
        if (!ok && navigator.clipboard && navigator.clipboard.writeText) {
          navigator.clipboard.writeText(text).then(
            function(){ btn.innerText = '✓ Copied!'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); },
            function(){ btn.innerText = 'Press Ctrl+C'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); }
          );
        } else {
          btn.innerText = ok ? '✓ Copied!' : 'Press Ctrl+C';
          setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
        }
      "
      style="background: #1B5162; color: #FFFFFF; border: none; border-radius: 6px; padding: 6px 18px; font-size: 11pt; font-weight: 700; cursor: pointer; white-space: nowrap; transition: background 0.15s ease;">Copy</button>
  </div>
  <pre style="margin: 0; padding: 16px 18px; background: #FFFFFF; color: #0b2026; font-family: Menlo, Consolas, monospace; font-size: 12pt; line-height: 1.55; white-space: pre-wrap; word-wrap: break-word; overflow-x: auto;">
Using the participants_bronze table:
- Drop the _rescued_data column.
- Cast join_time and leave_time from STRING to TIMESTAMP.
- Drop all rows with a `NULL` id.
- Create a new numeric column named duration_in_minutes that is the difference between leave_time and join_time in minutes, rounded to a whole number. If leave_time is before join_time (the difference would be negative), set duration_in_minutes to NULL instead.
- Keep all other columns.
- Create a new table in the same catalog and schema named participants_silver.
  </pre>
</div>



3. **Genie Code** generates multiple nodes in the canvas. Explore each node.
    > **Notice that the AI inferred the operators and expressions it needed. Always inspect what was generated before you trust it.**

4. Review each node that was generated (the configuration and output panel). Then click **Accept all** if everything looks as expected.

5. Select the **Output** node (the last node in the chain) and view its **Preview** pane. Confirm:
    - **join_time** and **leave_time** are `TIMESTAMP` types.
    - **duration_in_minutes** exists and contains whole numbers.
    - **_rescued_data** is no longer in the column list.

    > **NOTE:** Use the checkpoint below to compare your result against the expected output.

6. If the result is not what you expected, you can:
    - **Manually adjust** the generated node in its configuration panel.
    - **Manually edit** the AI generated description to regenerate the operator.
    - **Re-prompt Genie Code** with a refined description.

7. Rename the generated **Output** node to `participants_silver_table`.

8. Select **Run** on the **Output** node to create the `participants_silver` table in Unity Catalog.
    - Confirm the run was successful.

### Checkpoint - Participants Silver Table

![participants_silver_table](./Includes/images/visual_etl_pipeline/checkpoint_participants_silver-tbl.png)

## C. Create the `participant_lookup_silver` Table

Once again we will use **Genie Code** (AI) to build this silver table in a single prompt. This time the focus is on **string parsing**: splitting one column into two.

In the data we need to:

- Drop the **_rescued_data** column.
- Split the **email** column into **email_username** and **email_domain** (the parts before and after `@`).
- Split the **region** column into **geo** and **sub_region** (the parts before and after `-`).
- Keep all other columns as is, including the original **email** and **region**.

1. Select the **participant_lookup_bronze** node and view the data in the **Preview** pane.

2. In the **Genie Code** prompt bar, enter the following prompt:

<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; margin: 8px 0; font-family: sans-serif; max-width: 1100px; overflow: hidden;">
  <div style="display: flex; justify-content: space-between; align-items: center; background: #EEEDE9; padding: 6px 12px; border-bottom: 1px solid #d9d7d2;">
    <span style="font-size: 10pt; font-weight: 700; color: #0b2026; letter-spacing: 1px; text-transform: uppercase;">Genie Code Prompt</span>
    <button
      type="button"
      aria-label="Copy Genie Code prompt to clipboard"
      onmouseover="this.style.background='#143F4D';"
      onmouseout="this.style.background='#1B5162';"
      onclick="
        var text = this.parentElement.nextElementSibling.innerText;
        var btn = this;
        var ok = false;
        try {
          var ta = document.createElement('textarea');
          ta.value = text;
          ta.style.position = 'fixed';
          ta.style.top = '0';
          ta.style.left = '0';
          ta.style.opacity = '0';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          ok = document.execCommand('copy');
          document.body.removeChild(ta);
        } catch (e) { ok = false; }
        if (!ok && navigator.clipboard && navigator.clipboard.writeText) {
          navigator.clipboard.writeText(text).then(
            function(){ btn.innerText = '✓ Copied!'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); },
            function(){ btn.innerText = 'Press Ctrl+C'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); }
          );
        } else {
          btn.innerText = ok ? '✓ Copied!' : 'Press Ctrl+C';
          setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
        }
      "
      style="background: #1B5162; color: #FFFFFF; border: none; border-radius: 6px; padding: 6px 18px; font-size: 11pt; font-weight: 700; cursor: pointer; white-space: nowrap; transition: background 0.15s ease;">Copy</button>
  </div>
  <pre style="margin: 0; padding: 16px 18px; background: #FFFFFF; color: #0b2026; font-family: Menlo, Consolas, monospace; font-size: 12pt; line-height: 1.55; white-space: pre-wrap; word-wrap: break-word; overflow-x: auto;">
Using the participant_lookup_bronze table:
- Drop the _rescued_data column.
- Split the email column into two new columns: email_username (the part before the @ symbol) and email_domain (the part after the @ symbol).
- Split the region column into two new columns: geo (the part before the hyphen) and sub_region (the part after the hyphen). If there is only one value in Region it should be the region, and the sub region should be empty.
- Keep all other columns, including the original email and region columns.
- Create a new table in the same catalog and schema named participant_lookup_silver.
  </pre>
</div>



3. **Genie Code** generates nodes in the canvas. Explore each node.
    > **As before, inspect what was generated before you trust it.**

4. Review each node that was generated (the configuration and output panel). Then click **Accept all** if everything looks as expected.

5. Select the **Output** node (the last node in the chain) and view its **Preview** pane. Confirm:
    - **email_username** and **email_domain** exist and look correct (for example, `user_0373` and `contosoltd.com`).
    - **geo** and **sub_region** exist and look correct (for example, `EMEA` and `North`).
    - **email** and **region** are still in the column list.
    - **_rescued_data** is no longer in the column list.

    > **NOTE:** Use the checkpoint below to compare your result against the expected output.

6. If the result is not what you expected, you can:
    - **Manually adjust** the generated node in its configuration panel.
    - **Manually edit** the AI generated description to regenerate the operator.
    - **Re-prompt Genie Code** with a refined description.

7. Rename the generated **Output** node to `participant_lookup_silver_table`.

8. Select **Run** on the **Output** node to create the `participant_lookup_silver` table in Unity Catalog. Confirm the run was successful.

### Checkpoint - Participant Lookup Silver Table

![participant_lookup_silver_table](./Includes/images/visual_etl_pipeline/checkpoint-participants-lookup-silver-tbl.png)

## D. Create the `feedback_silver` Table

This is the most challenging silver table because of the nested JSON inside the **responses** column.

The **feedback_bronze** table has only two columns:
- **meeting_uuid**
- **responses**

For this silver table we will use the **SQL** operator to parse and flatten the JSON.


### D1. What is in the Responses Column?

- Each row in **feedback_bronze** contains one **meeting_uuid** and a `STRING` that looks like the JSON below.

- Each object in the array is one answer from one respondent to one question.

- Every object has 4 fields: **answer**, **question**, **question_id**, **respondent_id**.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- Side-by-side: JSON left, annotations right -->
<div style="display: grid; grid-template-columns: minmax(0, 1fr) 320px; gap: 5px; align-items: start; margin-bottom: 20px;">

  <!-- Left: syntax-highlighted JSON -->
  <div style="min-width: 0;">
    <div style="font-size: 11pt; font-weight: 700; color: #618794; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 8px;">Sample responses value</div>
    <div class="code-block" data-language="json">
[
  {
    "answer": "4",
    "question": "Rate the session content.",
    "question_id": "UXmnHGWqQ5KLINqgt2D7sA",
    "respondent_id": "ze8gFlIwQOasa1s6JVi+4w=="
  },
  {
    "answer": "I think finally understand how all the pieces fit together end to end",
    "question": "What did you like about the content in this session?",
    "question_id": "r6Wd31x8THycYxzVzeZ0BA",
    "respondent_id": "ze8gFlIwQOasa1s6JVi+4w=="
  },
  {
    ....
  }
]
    </div>
  </div>

  <!-- Right: field annotations -->
  <div style="min-width: 0; overflow: hidden; display: flex; flex-direction: column; gap: 2px;">
    <div style="font-size: 11pt; font-weight: 700; color: #618794; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 2px;">The 4 fields in each object</div>
    <div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; padding: 5px 12px; font-size: 13pt; line-height: 1.4; overflow-wrap: anywhere; word-wrap: break-word; box-sizing: border-box; width: 100%;">
      <span style="font-weight: 700; color: #1B5162; font-family: Consolas, Monaco, monospace;">answer</span>
      <span style="color: #5E7077;"> Always a string, even ratings.</span>
    </div>
    <div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; padding: 5px 12px; font-size: 13pt; line-height: 1.4; overflow-wrap: anywhere; word-wrap: break-word; box-sizing: border-box; width: 100%;">
      <span style="font-weight: 700; color: #1B5162; font-family: Consolas, Monaco, monospace;">question</span>
      <span style="color: #5E7077;"> Human-readable question text.</span>
    </div>
    <div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; padding: 5px 12px; font-size: 13pt; line-height: 1.4; overflow-wrap: anywhere; word-wrap: break-word; box-sizing: border-box; width: 100%;">
      <span style="font-weight: 700; color: #1B5162; font-family: Consolas, Monaco, monospace;">question_id</span>
      <span style="color: #5E7077;"> Stable id used to filter by question.</span>
    </div>
    <div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; padding: 5px 12px; font-size: 13pt; line-height: 1.4; overflow-wrap: anywhere; word-wrap: break-word; box-sizing: border-box; width: 100%;">
      <span style="font-weight: 700; color: #1B5162; font-family: Consolas, Monaco, monospace;">respondent_id</span>
      <span style="color: #5E7077;"> Attendee who submitted the answer.</span>
    </div>
  </div>

</div>

<!-- Footer callout -->
<div style="background: #FFF6F4; border: 2px solid #FF5F46; border-radius: 10px; padding: 16px 20px; font-size: 14pt; color: #0b2026; line-height: 1.55;">
  <strong style="color: #98102A;">Why answer is always a STRING:</strong> the same field holds numeric ratings like <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px;">"4"</code> and free-text feedback in the same column. We keep it as <strong>STRING</strong> in silver and cast it to <strong>INT</strong> only for the rating questions later in gold.
</div>

</div>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'json';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<pre style="background:#f8f8f8;border-radius:8px;padding:14px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:12.5pt;"></code></pre>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
    });
})();
</script>


### D2. Before and After
<br></br>
<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- BEFORE: feedback_bronze -->
<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 10px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 16px 20px; position: relative; overflow: hidden;">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #CD7F32;"></div>

  <div style="margin: 6px 0 14px 0;">
    <div style="font-size: 11pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #CD7F32; margin-bottom: 4px;">Before · Bronze</div>
    <div style="font-size: 15pt; font-weight: 700; color: #0b2026; font-family: Consolas, Monaco, monospace; margin-bottom: 2px;">feedback_bronze</div>
    <div style="font-size: 12pt; color: #618794;">1 row per session. The <strong>responses</strong> column holds a JSON array of every answer for that session.</div>
  </div>

  <div style="overflow-x: auto;">
    <table style="width: 100%; border-collapse: collapse; font-size: 13pt; min-width: 600px;">
      <thead>
        <tr style="background: #EEEDE9;">
          <th style="text-align: left; padding: 8px 12px; font-weight: 700; color: #0b2026; border: 1px solid #DCDCDC; width: 200px;">meeting_uuid</th>
          <th style="text-align: left; padding: 8px 12px; font-weight: 700; color: #0b2026; border: 1px solid #DCDCDC;">responses</th>
        </tr>
      </thead>
      <tbody>
        <tr style="background: #ffffff;">
          <td style="padding: 8px 12px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077; vertical-align: top;">/e3hcn3...</td>
          <td style="padding: 8px 12px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077; vertical-align: top;">[{"answer":"4","question":"Rate the session content.",...}, {"answer":"3","question":"Rate the speaker.",...}, {"answer":"I think finally...",...}, ...]</td>
        </tr>
        <tr style="background: #ffffff;">
          <td style="padding: 8px 12px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077; vertical-align: top;">n72+ES...</td>
          <td style="padding: 8px 12px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077; vertical-align: top;">[{"answer":"2","question":"Rate the session content.",...}, {"answer":"5","question":"Rate the speaker.",...}, ...]</td>
        </tr>
      </tbody>
    </table>
  </div>
</div>

<!-- Arrow with caption -->
<div style="text-align: center; margin: 14px 0;">
  <svg width="28" height="36" viewBox="0 0 28 36" fill="none" style="display: inline-block;">
    <line x1="14" y1="2" x2="14" y2="24" stroke="#5E7077" stroke-width="3" stroke-linecap="round"/>
    <polygon points="4,24 24,24 14,36" fill="#5E7077"/>
  </svg>
  <div style="font-size: 13pt; font-weight: 700; color: #1B5162; margin-top: 4px;">Parse JSON to VARIANT · Explode the array · Extract fields</div>
</div>

<!-- AFTER: feedback_silver -->
<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 10px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 16px 20px; position: relative; overflow: hidden;">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #90A5B1;"></div>

  <div style="margin: 6px 0 14px 0;">
    <div style="font-size: 11pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #5A6F77; margin-bottom: 4px;">After · Silver</div>
    <div style="font-size: 15pt; font-weight: 700; color: #0b2026; font-family: Consolas, Monaco, monospace; margin-bottom: 2px;">feedback_silver</div>
    <div style="font-size: 12pt; color: #618794;">1 row per individual response. Each field from the JSON is now its own column.</div>
  </div>

  <div style="overflow-x: auto;">
    <table style="width: 100%; border-collapse: collapse; font-size: 12.5pt; min-width: 750px;">
      <thead>
        <tr style="background: #EEEDE9;">
          <th style="text-align: left; padding: 8px 10px; font-weight: 700; color: #0b2026; border: 1px solid #DCDCDC;">meeting_uuid</th>
          <th style="text-align: left; padding: 8px 10px; font-weight: 700; color: #0b2026; border: 1px solid #DCDCDC;">answer</th>
          <th style="text-align: left; padding: 8px 10px; font-weight: 700; color: #0b2026; border: 1px solid #DCDCDC;">question</th>
          <th style="text-align: left; padding: 8px 10px; font-weight: 700; color: #0b2026; border: 1px solid #DCDCDC;">question_id</th>
          <th style="text-align: left; padding: 8px 10px; font-weight: 700; color: #0b2026; border: 1px solid #DCDCDC;">respondent_id</th>
        </tr>
      </thead>
      <tbody>
        <tr style="background: #ffffff;">
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">/e3hcn3...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #0b2026;">4</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; color: #0b2026;">Rate the session content.</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">UXmnHGW...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">ze8gFlIw...</td>
        </tr>
        <tr style="background: #ffffff;">
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">/e3hcn3...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #0b2026;">3</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; color: #0b2026;">Rate the speaker.</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">rplPVIaI...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">ze8gFlIw...</td>
        </tr>
        <tr style="background: #ffffff;">
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">/e3hcn3...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; color: #0b2026;">I think finally understand how all the pieces fit together...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; color: #0b2026;">What did you like about the content?</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">r6Wd31x8...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">ze8gFlIw...</td>
        </tr>
        <tr style="background: #ffffff;">
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">/e3hcn3...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #0b2026;">4</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; color: #0b2026;">Rate the session content.</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">UXmnHGW...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">So1HPj3v...</td>
        </tr>
        <tr style="background: #ffffff;">
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">/e3hcn3...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #0b2026;">5</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; color: #0b2026;">Rate the speaker.</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">rplPVIaI...</td>
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; font-family: Consolas, Monaco, monospace; color: #5E7077;">So1HPj3v...</td>
        </tr>
        <tr style="background: #ffffff;">
          <td style="padding: 7px 10px; border: 1px solid #EEEDE9; color: #C2C2C2; text-align: center;" colspan="5">...many more rows...</td>
        </tr>
      </tbody>
    </table>
  </div>
</div>

<!-- Takeaway -->
<div style="margin-top: 16px; background: #F8F9FC; border-left: 6px solid #1B5162; border-radius: 8px; padding: 14px 18px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.55;">
    <strong>One bronze row (meeting uuid) becomes many silver rows.</strong> Each answer now has its own row with proper columns, ready to filter, join, and aggregate downstream.
  </div>
</div>

</div>


### D3. Explode the Array of Nested JSON

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">
You could have asked <strong>Genie Code</strong> to write this SQL for you, just like we did in other nodes. We're using the <strong>SQL</strong> operator directly here so you can explore other operators within Lakeflow Designer.
  </div>
</div>


1. Select the **feedback_bronze** node and view the data in the **Preview** pane.

2. Select **+** > **SQL** to add the SQL operator and connect it to **feedback_bronze**.

3. In the **SQL** editor, replace the default query with the following:

<div class="code-block" data-language="sql">
SELECT
    r.meeting_uuid,
    e.value:answer::STRING        AS answer,
    e.value:question::STRING      AS question,
    e.value:question_id::STRING   AS question_id,
    e.value:respondent_id::STRING AS respondent_id
FROM feedback_bronze r
CROSS JOIN LATERAL variant_explode_outer(parse_json(r.responses)) AS e
</div>

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    This single query does three things at once:
  </strong>
  <div style="color:#333;">

- `parse_json(r.responses)` converts the **STRING** column into a **VARIANT**.
- `variant_explode_outer(...)` joined with `CROSS JOIN LATERAL` produces one row per element of the VARIANT array, exposing each element as `e.value`.
- `e.value:field::STRING` uses VARIANT path access (`:`) and type cast (`::`) to pull each field out as a `STRING` column.

  </div>
</div>


4. Click **Apply**.

5. View the **Preview** pane and confirm:
    - The columns are now **meeting_uuid**, **answer**, **question**, **question_id**, **respondent_id**.
    - Each column is a `STRING`.
    - The row count is much higher than **feedback_bronze** (40 rows in bronze, 1,000+ in silver).

6. Rename the SQL node to `feedback_silver_explode`.

7. View and compare the **Input** and **Output** previews.

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:0;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12pt;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:14px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13pt;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            var self = this;
            setTimeout(function() { self.textContent = 'Copy'; }, 2000);
        };
    });
})();
</script>


### D4. Create the Silver Table

1. Select the **feedback_silver_explode** node.

2. In the **Genie Code** prompt bar, enter the following prompt:

<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; margin: 8px 0; font-family: sans-serif; max-width: 1100px; overflow: hidden;">
  <div style="display: flex; justify-content: space-between; align-items: center; background: #EEEDE9; padding: 6px 12px; border-bottom: 1px solid #d9d7d2;">
    <span style="font-size: 10pt; font-weight: 700; color: #0b2026; letter-spacing: 1px; text-transform: uppercase;">Genie Code Prompt</span>
    <button
      type="button"
      aria-label="Copy Genie Code prompt to clipboard"
      onmouseover="this.style.background='#143F4D';"
      onmouseout="this.style.background='#1B5162';"
      onclick="
        var text = this.parentElement.nextElementSibling.innerText;
        var btn = this;
        var ok = false;
        try {
          var ta = document.createElement('textarea');
          ta.value = text;
          ta.style.position = 'fixed';
          ta.style.top = '0';
          ta.style.left = '0';
          ta.style.opacity = '0';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          ok = document.execCommand('copy');
          document.body.removeChild(ta);
        } catch (e) { ok = false; }
        if (!ok && navigator.clipboard && navigator.clipboard.writeText) {
          navigator.clipboard.writeText(text).then(
            function(){ btn.innerText = '✓ Copied!'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); },
            function(){ btn.innerText = 'Press Ctrl+C'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); }
          );
        } else {
          btn.innerText = ok ? '✓ Copied!' : 'Press Ctrl+C';
          setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
        }
      "
      style="background: #1B5162; color: #FFFFFF; border: none; border-radius: 6px; padding: 6px 18px; font-size: 11pt; font-weight: 700; cursor: pointer; white-space: nowrap; transition: background 0.15s ease;">Copy</button>
  </div>
  <pre style="margin: 0; padding: 16px 18px; background: #FFFFFF; color: #0b2026; font-family: Menlo, Consolas, monospace; font-size: 12pt; line-height: 1.55; white-space: pre-wrap; word-wrap: break-word; overflow-x: auto;">
Create a table named feedback_silver in my designer_meeting schema
  </pre>
</div>



3. Review the **Output** node configuration that Genie Code generated (table name, catalog, schema). Compare it against the checkpoint image below.

4. Name the node `feedback_silver_table`.

5. Select **Run** to create the `feedback_silver` table and view the **Output**.

#### Checkpoint - Feedback Silver Table

![feedback_silver_table](./Includes/images/visual_etl_pipeline/checkpoint-silver-feedback-tbl.png)

## E. Auto Layout the Nodes

1. In the main Lakeflow Designer toolbar, select the **Auto Layout** ![Layout](./Includes/images/common/auto-layout-icon.png) icon to organize the nodes in the canvas.

#### Checkpoint - Feedback Bronze - Silver Visual Workflow

![feedback_silver_table](./Includes/images/visual_etl_pipeline/checkpoint-file-silver-layout.png)




&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>